In [1]:
import os
import numpy as np
import h5py
from scipy import signal
from scipy.stats import pearsonr
import seaborn as sns
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import string

# Define path to data:
script_dir = os.path.dirname(os.path.abspath("Figure 1 Data Summary.ipynb"))
parent_dir = os.path.dirname(script_dir)
data_path = (parent_dir + '/dfs/')

In [5]:
# Load cshq data file:
df_cshq = pd.read_csv(data_path+'df_cshq_git.csv')


print(len(df_cshq))

200


In [6]:
# Convert 'SO_weekdays' and 'SO_weekends' to datetime
df_cshq['SO_weekdays'] = pd.to_datetime(df_cshq['SO_weekdays'], format='%I:%M %p')
df_cshq['SO_weekends'] = pd.to_datetime(df_cshq['SO_weekends'], format='%I:%M %p')

df_cshq['SO_weekdays_Relative'] = df_cshq['SO_weekdays'].dt.hour * 60 + df_cshq['SO_weekdays'].dt.minute
df_cshq['SO_weekends_Relative'] = df_cshq['SO_weekends'].dt.hour * 60 + df_cshq['SO_weekends'].dt.minute


df_cshq['SO_weekdays_Relative'] = np.where(df_cshq['SO_weekdays_Relative'] < 12*60,
                                           df_cshq['SO_weekdays_Relative'] + 24*60,
                                           df_cshq['SO_weekdays_Relative'])

df_cshq['SO_weekends_Relative'] = np.where(df_cshq['SO_weekends_Relative'] < 12*60,
                                           df_cshq['SO_weekends_Relative'] + 24*60,
                                           df_cshq['SO_weekends_Relative'])

df_cshq['SO'] = (df_cshq['SO_weekdays_Relative'] + df_cshq['SO_weekends_Relative']) / 2
df_cshq['SO'] = pd.to_datetime(df_cshq['SO'], unit='m').dt.strftime('%H:%M')

df_cshq['SO_FromMidNight'] = (df_cshq['SO_weekdays_Relative'] + df_cshq['SO_weekends_Relative']) / 2


# Remove _Relative columns:
df_cshq = df_cshq.drop(columns=['SO_weekdays_Relative', 'SO_weekends_Relative'])


# Convert 'FA_weekdays' and 'SO_weekends' to datetime
df_cshq['FA_weekdays'] = pd.to_datetime(df_cshq['FA_weekdays'], format='%I:%M %p')
df_cshq['FA_weekends'] = pd.to_datetime(df_cshq['FA_weekends'], format='%I:%M %p')

df_cshq['FA_weekdays_Relative'] = df_cshq['FA_weekdays'].dt.hour * 60 + df_cshq['FA_weekdays'].dt.minute
df_cshq['FA_weekends_Relative'] = df_cshq['FA_weekends'].dt.hour * 60 + df_cshq['FA_weekends'].dt.minute

df_cshq['FA'] = (df_cshq['FA_weekdays_Relative'] + df_cshq['FA_weekends_Relative']) / 2
df_cshq['FA'] = pd.to_datetime(df_cshq['FA'], unit='m').dt.strftime('%H:%M')

df_cshq['FA_FromMidNight'] = ((df_cshq['FA_weekdays_Relative'] + df_cshq['FA_weekends_Relative']) / 2)+24*60

# Remove _Relative columns:
df_cshq = df_cshq.drop(columns=['FA_weekdays_Relative', 'FA_weekends_Relative'])


# Remove weekday and weekend columns:
df_cshq = df_cshq.drop(columns=['SO_weekdays', 'SO_weekends', 'FA_weekdays', 'FA_weekends'])

# Change to hours:
df_cshq['SO_FromMidNight'] = df_cshq['SO_FromMidNight']/60
df_cshq['FA_FromMidNight'] = df_cshq['FA_FromMidNight']/60

In [7]:
# Save to csv:
df_cshq.to_csv(data_path+'df_cshq_clean.csv', index=False)